# `water_quality` 02: noteworthy single-feature findings

**Purpose:** identify and discuss supported points that stand out after the
standard `category` breakdown. Target relationships here are
exploratory and must be rechecked after the split is frozen.


In [1]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display


def find_stage_directory():
    start = Path.cwd().resolve()
    for candidate in (start, *start.parents):
        if (
            (candidate / "data" / "TrainingSetValues.csv").exists()
            and (candidate / "src" / "source_data_validation.py").exists()
        ):
            return candidate
    raise FileNotFoundError("Could not locate the stage-1-pump-it-up directory.")


stage_directory = find_stage_directory()
source_directory = str((stage_directory / "src").resolve())
if source_directory not in sys.path:
    sys.path.insert(0, source_directory)

from predictor_audit import (
    analysis_categories,
    categorical_summary,
    categorical_target_profile,
    category_frequency_table,
    numeric_summary,
    numeric_target_summary,
    related_feature_summary,
    sentinel_mask,
    source_blank_mask,
    text_normalisation_summary,
)
from source_data_validation import (
    validate_aligned_ids,
    validate_label_frame,
    validate_raw_feature_schema,
)

data_directory = stage_directory / "data"
training_features = pd.read_csv(
    data_directory / "TrainingSetValues.csv",
    keep_default_na=False,
)
training_labels = pd.read_csv(
    data_directory / "TrainingSetLabels.csv",
    keep_default_na=False,
)
test_features = pd.read_csv(
    data_directory / "TestSetValues.csv",
    keep_default_na=False,
)

validate_raw_feature_schema(training_features)
validate_raw_feature_schema(test_features)
validate_label_frame(training_labels)
validate_aligned_ids(training_features, training_labels)

training_data = training_features.merge(
    training_labels,
    on="id",
    validate="one_to_one",
)

feature = 'water_quality'
feature_metadata = {'order': 31, 'name': 'water_quality', 'audit_type': 'category', 'role': 'candidate', 'disposition': 'retain as granular water-quality representation', 'finding': 'Eight levels map deterministically to quality_group and preserve important within-group differences.', 'decision': 'Retain granular quality with rare handling and compare the coarse parent by ablation.', 'risk': 'The unknown state may reflect data quality as much as water quality.', 'sentinel_tokens': ['unknown'], 'related': [{'feature': 'quality_group', 'reason': 'This is the deterministic coarse parent.'}, {'feature': 'source', 'reason': 'Water source influences observed quality.'}, {'feature': 'quantity', 'reason': 'Water quality and availability jointly describe service state.'}]}
feature_types = {'amount_tsh': 'numeric', 'date_recorded': 'date', 'funder': 'high-cardinality-category', 'gps_height': 'numeric', 'installer': 'high-cardinality-category', 'longitude': 'coordinate', 'latitude': 'coordinate', 'wpt_name': 'high-cardinality-category', 'num_private': 'numeric', 'basin': 'category', 'subvillage': 'high-cardinality-category', 'region': 'category', 'region_code': 'category', 'district_code': 'category', 'lga': 'category', 'ward': 'high-cardinality-category', 'population': 'numeric', 'public_meeting': 'binary', 'recorded_by': 'constant', 'scheme_management': 'category', 'scheme_name': 'high-cardinality-category', 'permit': 'binary', 'construction_year': 'year', 'extraction_type': 'category', 'extraction_type_group': 'category', 'extraction_type_class': 'category', 'management': 'category', 'management_group': 'category', 'payment': 'category', 'payment_type': 'category', 'water_quality': 'category', 'quality_group': 'category', 'quantity': 'category', 'quantity_group': 'category', 'source': 'category', 'source_type': 'category', 'source_class': 'category', 'waterpoint_type': 'category', 'waterpoint_type_group': 'category'}
assert feature in training_features.columns
print(
    f"Validated {len(training_features):,} training rows and "
    f"{len(test_features):,} test rows for {feature}."
)


Validated 59,400 training rows and 14,850 test rows for water_quality.


## Supported target evidence


In [2]:
sentinel_tokens = ['unknown']
target_profile = categorical_target_profile(
    training_data,
    feature,
    minimum_support=100,
    sentinel_tokens=sentinel_tokens,
)
display(target_profile.head(20))

supported = target_profile.loc[target_profile["meets support threshold"]].copy()
non_functional_column = "non functional (%)"
if non_functional_column in supported:
    display(
        supported.sort_values(non_functional_column, ascending=False)
        .head(12)[["rows", non_functional_column]]
    )


status_group,rows,meets support threshold,functional (%),functional needs repair (%),non functional (%)
water_quality,,,,,
soft,50818,True,56.59,7.68,35.72
salty,4856,True,45.72,4.63,49.65
unknown,1876,True,14.07,1.87,84.06
milky,804,True,54.48,1.74,43.78
coloured,490,True,50.20,11.02,38.78
salty abandoned,339,True,51.33,21.24,27.43
fluoride,200,True,75.50,6.50,18.00
fluoride abandoned,17,False,35.29,0.00,64.71


status_group,rows,non functional (%)
water_quality,,
unknown,1876,84.06
salty,4856,49.65
milky,804,43.78
coloured,490,38.78
soft,50818,35.72
salty abandoned,339,27.43
fluoride,200,18.00


## Observation

Eight levels map deterministically to quality_group and preserve important within-group differences.

## Interpretation

The supported single-feature patterns make this field worth the stated
treatment, but they do not prove causation or independent predictive value.
High-cardinality and geographic fields are especially vulnerable to
memorisation under a random split.

## Provisional decision

Retain granular quality with rare handling and compare the coarse parent by ablation.

**Risk to carry forward:** The unknown state may reflect data quality as much as water quality.


In [3]:
decision_record = pd.DataFrame([{
    "feature": feature,
    "role": feature_metadata["role"],
    "disposition": feature_metadata["disposition"],
    "finding": feature_metadata["finding"],
    "decision": feature_metadata["decision"],
    "risk": feature_metadata["risk"],
}])
display(decision_record.set_index("feature"))


,role,disposition,finding,decision,risk
feature,,,,,
water_quality,candidate,retain as granular water-quality representation,Eight levels map deterministically to quality_...,Retain granular quality with rare handling and...,The unknown state may reflect data quality as ...
